# Information Security Lab
### Anyuta Kumar
### October 12,2026
## LAB 8: Searchable Encryption

1. Execute the following for SSE:

  a. Create a dataset: Generate a text corpus of at least ten documents. Each document
should contain multiple words.

  b. Implement encryption and decryption functions: Use the AES encryption and
decryption functions.

  c. Create an inverted index: Build an inverted index mapping word to the list of
document IDs containing those words.
- Encrypt the index using the provided encryption function.

  d. Implement the search function:
- Take a search query as input.
- Encrypt the query.
- Search the encrypted index for matching terms.
- Decrypt the returned document IDs and display the corresponding documents

In [1]:
import re
import json
import base64
import hashlib
from cryptography.fernet import Fernet

documents = {
    1: "Python is a popular programming language",
    2: "Java is used for enterprise application development",
    3: "Python suppors machine learning and data science",
    4: "Artificial intelligence uses machine learning algorithms",
    5: "Data science involves statistics and visualization",
    6: "Cybersecurity protects networks and computer systems",
    7: "Encryption protects sensitive data and information",
    8: "Cloud computing provides scalable computing resources",
    9: "Computer networks enable communication between systems",
    10: "Python programming supports artificial intelligence"
}
print("DATASET")
for doc_id, text in documents.items():
    print(f"Document {doc_id}: {text}")
key = Fernet.generate_key()
cipher = Fernet(key)
def encrypt(data):
    """Encrypt a string using Fernet (AES-based encryption)."""
    if not isinstance(data, str):
        data = json.dumps(data, sort_keys=True)
    return cipher.encrypt(data.encode()).decode()
def decrypt(encrypted_data):
    """Decrypt ciphertext and return the original string."""
    return cipher.decrypt(encrypted_data.encode()).decode()
def tokenize(text):
    return re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
inverted_index = {}
for doc_id, text in documents.items():
    for word in set(tokenize(text)):
        inverted_index.setdefault(word, []).append(doc_id)
print("\nPLAIN INVERTED INDEX")
print(json.dumps(inverted_index, indent=2))
encrypted_index = {}
for word, doc_ids in inverted_index.items():
    encrypted_word = encrypt(word)
    encrypted_doc_ids = encrypt(json.dumps(doc_ids))
    encrypted_index[encrypted_word] = encrypted_doc_ids
print("\nENCRYPTED INVERTED INDEX")
print(json.dumps(encrypted_index, indent=2))
def search(query):
    print(f"\nSearch query: {query}")
    query_words = tokenize(query)
    if not query_words:
        print("Please enter a valid search query.")
        return
    encrypted_query = [encrypt(word) for word in query_words]
    print("\nEncrypted query terms:")
    for item in encrypted_query:
        print(item)
    matching_doc_ids = set()
    for encrypted_word, encrypted_ids in encrypted_index.items():
        stored_word = decrypt(encrypted_word)
        if stored_word in query_words:
            doc_ids = json.loads(decrypt(encrypted_ids))
            matching_doc_ids.update(doc_ids)
    if matching_doc_ids:
        print("\nMatching documents:")
        for doc_id in sorted(matching_doc_ids):
            print(f"Document {doc_id}: {documents[doc_id]}")
    else:
        print("No matching documents found.")
search("Python")
search("machine learning")
search("computer networks")
search("encryption data")

DATASET
Document 1: Python is a popular programming language
Document 2: Java is used for enterprise application development
Document 3: Python suppors machine learning and data science
Document 4: Artificial intelligence uses machine learning algorithms
Document 5: Data science involves statistics and visualization
Document 6: Cybersecurity protects networks and computer systems
Document 7: Encryption protects sensitive data and information
Document 8: Cloud computing provides scalable computing resources
Document 9: Computer networks enable communication between systems
Document 10: Python programming supports artificial intelligence

PLAIN INVERTED INDEX
{
  "python": [
    1,
    3,
    10
  ],
  "is": [
    1,
    2
  ],
  "programming": [
    1,
    10
  ],
  "a": [
    1
  ],
  "language": [
    1
  ],
  "popular": [
    1
  ],
  "development": [
    2
  ],
  "enterprise": [
    2
  ],
  "application": [
    2
  ],
  "used": [
    2
  ],
  "for": [
    2
  ],
  "java": [
    2
 

2. Execute the following for PKSE:

  a. Create a dataset:
- Generate a text corpus of at least ten documents. Each document should contain
multiple words.

  b. Implement encryption and decryption functions:
-  Use the Paillier cryptosystem for encryption and decryption.
-
  c. Create an encrypted index:
-  Build an inverted index mapping word to the list of document IDs containing
those words.
- Encrypt the index using the Paillier cryptosystem.
-
  d. Implement the search function:
- Take a search query as input.
- Encrypt the query using the public key.
- Search the encrypted index for matching terms.
- Decrypt the returned document IDs using the private key.

In [2]:
import re
import json
from phe import paillier

documents = {
    1: "Python is a popular programming language",
    2: "Java is used for enterprise application development",
    3: "Python supports machine learning and data science",
    4: "Artificial intelligence uses machine learning algorithms",
    5: "Data science involves statistics and visualization",
    6: "Cybersecurity protects networks and computer systems",
    7: "Encryption protects sensitive data and information",
    8: "Cloud computing provides scalable computing resources",
    9: "Computer networks enable communication between systems",
    10: "Python programming supports artificial intelligence"
}
print("DATASET")
for doc_id, text in documents.items():
    print(f"Document {doc_id}: {text}")
public_key, private_key = paillier.generate_paillier_keypair(
    n_length=2048
)
def encrypt_data(data):
    """Encrypt an integer using the public key."""
    return public_key.encrypt(int(data))
def decrypt_data(encrypted_data):
    """Decrypt an encrypted integer using the private key."""
    return private_key.decrypt(encrypted_data)
def tokenize(text):
    return re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
inverted_index = {}
for doc_id, text in documents.items():
    for word in set(tokenize(text)):
        inverted_index.setdefault(word, []).append(doc_id)
print("\nPLAIN INVERTED INDEX")
print(json.dumps(inverted_index, indent=2))
vocabulary = sorted(inverted_index.keys())
word_to_id = {
    word: i + 1 for i, word in enumerate(vocabulary)
}
id_to_word = {
    i + 1: word for i, word in enumerate(vocabulary)
}
encrypted_index = {}
for word, doc_ids in inverted_index.items():
    word_id = word_to_id[word]
    encrypted_word_id = encrypt_data(word_id)
    encrypted_doc_ids = [
        encrypt_data(doc_id) for doc_id in doc_ids
    ]
    encrypted_index[word] = {
        "encrypted_word_id": encrypted_word_id,
        "encrypted_doc_ids": encrypted_doc_ids
    }
print("\nENCRYPTED INDEX")
print("Index encrypted successfully.")
print("Number of indexed terms:", len(encrypted_index))
def search(query):
    print("\nSearch query:", query)
    query_words = set(tokenize(query))
    if not query_words:
        print("Please enter a valid query.")
        return
    encrypted_query = {
        word: encrypt_data(word_to_id[word])
        for word in query_words
        if word in word_to_id
    }
    print("\nQuery encrypted using Paillier public key.")
    matching_doc_ids = set()
    for word, query_ciphertext in encrypted_query.items():
        query_word_id = decrypt_data(query_ciphertext)

        for stored_word, entry in encrypted_index.items():
            stored_word_id = decrypt_data(
                entry["encrypted_word_id"]
            )
            if query_word_id == stored_word_id:
                for encrypted_doc_id in entry["encrypted_doc_ids"]:
                    doc_id = decrypt_data(encrypted_doc_id)
                    matching_doc_ids.add(doc_id)
    if matching_doc_ids:
        print("\nMatching document IDs:",
              sorted(matching_doc_ids))
        print("\nMATCHING DOCUMENTS")
        for doc_id in sorted(matching_doc_ids):
            print(f"Document {doc_id}: {documents[doc_id]}")
    else:
        print("No matching documents found.")
search("Python")
search("machine learning")
search("computer networks")
search("encryption data")
search("blockchain")

DATASET
Document 1: Python is a popular programming language
Document 2: Java is used for enterprise application development
Document 3: Python supports machine learning and data science
Document 4: Artificial intelligence uses machine learning algorithms
Document 5: Data science involves statistics and visualization
Document 6: Cybersecurity protects networks and computer systems
Document 7: Encryption protects sensitive data and information
Document 8: Cloud computing provides scalable computing resources
Document 9: Computer networks enable communication between systems
Document 10: Python programming supports artificial intelligence

PLAIN INVERTED INDEX
{
  "python": [
    1,
    3,
    10
  ],
  "is": [
    1,
    2
  ],
  "programming": [
    1,
    10
  ],
  "a": [
    1
  ],
  "language": [
    1
  ],
  "popular": [
    1
  ],
  "development": [
    2
  ],
  "enterprise": [
    2
  ],
  "application": [
    2
  ],
  "used": [
    2
  ],
  "for": [
    2
  ],
  "java": [
    2
